# Notebook 02 — Preprocesamiento NLP y Análisis Exploratorio de Textos

---

## TFM: Integración de Algoritmos de Aprendizaje Profundo para el Análisis de Sentimiento en Mercados Financieros

**Universidad Internacional de La Rioja (UNIR)**  
**Autores:** Armas Silva Stiv Alexis | Armas Silva Jonathan Fernando | Requelme Campoverde Adrian Alexander

---

## Objetivo de este Notebook

Este notebook cubre la **Fase 2: Preprocesamiento de Texto y EDA (Análisis Exploratorio de Datos) de Textos Financieros**.

Al finalizar este notebook habremos:

1. Cargado y explorado todos los datasets de texto disponibles
2. Aplicado un pipeline de limpieza NLP específico para textos financieros en español
3. Analizado la distribución estadística de los textos (longitud, idioma, fuente)
4. Generado nubes de palabras por categoría de sentimiento
5. Analizado las palabras más frecuentes por sentimiento (positivo/negativo/neutral)
6. Preparado un dataset limpio y listo para el entrenamiento de los modelos

---

## Marco Teórico — Preprocesamiento en NLP Financiero

El preprocesamiento de texto en el dominio financiero tiene características especiales:

- **Terminología específica**: Términos como 'bearish', 'halving', 'staking', 'yield' no están en diccionarios generales
- **Siglas y tickers**: \$BTC, ETH, SPY son entidades financieras que deben preservarse
- **Números críticos**: Porcentajes (+5%, -20%), precios (\$65,000) llevan información de sentimiento
- **Idioma mixto**: Muchos textos en español contienen términos técnicos en inglés ('bullish market', 'stop loss')
- **Ruido digital**: URLs, hashtags (#bitcoin), menciones (@usuario), emojis 🚀📉

La estrategia de limpieza debe **preservar la información de sentimiento** mientras elimina el ruido.

---
## Sección 1: Configuración del Entorno

In [1]:
# ============================================================
# CELDA 1: Importación de librerías para NLP
# ============================================================
# re          -> expresiones regulares para limpieza de texto
# nltk        -> toolkit NLP: tokenización, stopwords
# wordcloud   -> nubes de palabras para visualización
# collections -> Counter para contar frecuencias de palabras
# unicodedata -> normalización de caracteres especiales (tildes, ñ)

import warnings
warnings.filterwarnings('ignore')

import os, re, sys
import unicodedata
import sqlite3
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

import nltk

# Descargar recursos de NLTK (solo la primera vez)
for recurso in ['stopwords', 'punkt', 'punkt_tab', 'averaged_perceptron_tagger']:
    try:
        nltk.download(recurso, quiet=True)
    except:
        pass

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# ── Rutas del proyecto ────────────────────────────────────────
NOTEBOOK_DIR  = Path(os.path.abspath(''))
PROJECT_ROOT  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR

DATASETS_DIR  = PROJECT_ROOT / 'data' / 'datasets'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR   = PROJECT_ROOT / 'reports' / 'figures'
RAW_NEWS_DIR  = PROJECT_ROOT / 'data' / 'raw' / 'news'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Añadir src/ al path para importar nuestros módulos
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Configuración de gráficos
plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_columns', 10)

print('Librerias NLP cargadas correctamente')
print(f'Raiz del proyecto: {PROJECT_ROOT}')

Librerias NLP cargadas correctamente
Raiz del proyecto: C:\Users\StivArmas\OneDrive - ROBALINO\Documentos\MASTER\TFM\tfm-sentiment-financial


---
## Sección 2: Carga de Datasets de Texto

Trabajamos con tres fuentes de datos de texto:

| Dataset | Idioma | Registros | Fuente |
|---|---|---|---|
| Twitter Financial News | Inglés | 11,931 | HuggingFace (zeroshot) |
| Finance Sentiment ES (semilla) | Español | 30 | Manual — contexto Ecuador |
| Noticias BD SQLite | Español | Variable | El Comercio / Primicias |

> **Nota**: Los textos en inglés son útiles para entrenar y evaluar FinBERT (baseline). Los textos en español son el objetivo principal para el fine-tuning de BETO.

In [2]:
# ============================================================
# CELDA 2: Carga del dataset Twitter Financial News (inglés)
# ============================================================
# Este es el dataset principal para el baseline en inglés.
# 11,931 noticias financieras de Twitter etiquetadas como:
#   0 = bearish (negativo)   -> mapearemos a 'negative'
#   1 = bullish (positivo)   -> mapearemos a 'positive'
#   2 = neutral              -> mapearemos a 'neutral'

# Cargar el CSV
df_en = pd.read_csv(DATASETS_DIR / 'twitter_financial_news_en.csv')

# Mapear etiquetas numéricas a texto descriptivo
label_map = {0: 'negative', 1: 'positive', 2: 'neutral'}
df_en['sentiment'] = df_en['label'].map(label_map)
df_en['language']  = 'en'
df_en['source']    = 'Twitter Financial News'

print('=== DATASET: Twitter Financial News (Ingles) ===')
print(f'  Total registros: {len(df_en):,}')
print(f'  Columnas: {df_en.columns.tolist()}')
print()
print('Distribucion de sentimiento:')
dist = df_en['sentiment'].value_counts()
for sent, cnt in dist.items():
    pct = cnt / len(df_en) * 100
    barra = '#' * int(pct / 2)
    print(f'  {sent:10s}: {cnt:5,} ({pct:.1f}%) {barra}')

print()
print('Ejemplos por categoria:')
for sent in ['positive', 'negative', 'neutral']:
    ej = df_en[df_en['sentiment'] == sent]['text'].iloc[0]
    print(f'  [{sent.upper():8s}] {ej[:80]}...')

=== DATASET: Twitter Financial News (Ingles) ===
  Total registros: 11,931
  Columnas: ['text', 'label', 'label_text', 'sentiment', 'language', 'source']

Distribucion de sentimiento:
  neutral   : 7,744 (64.9%) ################################
  positive  : 2,398 (20.1%) ##########
  negative  : 1,789 (15.0%) #######

Ejemplos por categoria:
  [POSITIVE] $ALTG: Dougherty & Company starts at Buy...
  [NEGATIVE] $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT...
  [NEUTRAL ] $LB - MKM Partners puts a number on Victoria's Secret https://t.co/VSzHLqLBgE...


In [3]:
# ============================================================
# CELDA 3: Carga del dataset semilla en español
# ============================================================
# 30 ejemplos manuales en español con contexto ecuatoriano.
# Aunque pequeño, establece el vocabulario de referencia
# y la estructura de etiquetado para el fine-tuning.

df_es = pd.read_csv(DATASETS_DIR / 'finance_sentiment_es_seed.csv')
df_es['language'] = 'es'
df_es['source']   = 'Semilla Manual Ecuador'
# Renombrar columna de etiqueta para uniformidad
if 'label_text' in df_es.columns:
    df_es = df_es.rename(columns={'label_text': 'sentiment'})

print('=== DATASET: Semilla Español Ecuador ===')
print(f'  Total registros: {len(df_es):,}')
print()
print('Distribucion de sentimiento:')
dist_es = df_es['sentiment'].value_counts()
for sent, cnt in dist_es.items():
    print(f'  {sent:10s}: {cnt:3d} registros')

print()
print('Ejemplos de textos en español:')
for sent in ['positive', 'negative', 'neutral']:
    ej = df_es[df_es['sentiment'] == sent]['text'].iloc[0]
    print(f'  [{sent.upper():8s}] {ej}')

=== DATASET: Semilla Español Ecuador ===
  Total registros: 30

Distribucion de sentimiento:
  positive  :  10 registros
  negative  :  10 registros
  neutral   :  10 registros

Ejemplos de textos en español:
  [POSITIVE] Bitcoin alcanza nuevo máximo histórico y supera los $70,000 dólares
  [NEGATIVE] Bitcoin colapsa un 15% en las últimas 24 horas generando pánico en el mercado
  [NEUTRAL ] El Banco Central del Ecuador publica su informe trimestral de política monetaria


In [4]:
# ============================================================
# CELDA 4: Cargar noticias de la base de datos SQLite
# ============================================================
# La BD SQLite puede estar vacía si el scraper no ha recolectado
# noticias aún. En ese caso trabajamos con los datasets disponibles.

df_noticias = pd.DataFrame()
news_db_path = RAW_NEWS_DIR / 'news.db'

if news_db_path.exists():
    try:
        conn = sqlite3.connect(news_db_path)
        df_noticias = pd.read_sql_query(
            'SELECT title, summary, source, published_at, asset_related FROM news_articles LIMIT 5000',
            conn
        )
        conn.close()
        print(f'Noticias en BD SQLite: {len(df_noticias):,} registros')
        if len(df_noticias) > 0:
            print(df_noticias.head(3))
    except Exception as e:
        print(f'BD vacía o error: {e}')
else:
    print('BD de noticias no encontrada en:', news_db_path)

if len(df_noticias) == 0:
    print()
    print('INFO: La BD de noticias esta vacia.')
    print('      El scraper de El Comercio necesita ajuste de selectores CSS.')
    print('      Por ahora trabajamos con los datasets disponibles:')
    print(f'        - Twitter Financial News EN: {len(df_en):,} registros')
    print(f'        - Semilla Español Ecuador:   {len(df_es):,} registros')
    print()
    print('      En la seccion 7 generamos mas textos en español para ampliar el dataset.')

Noticias en BD SQLite: 0 registros

INFO: La BD de noticias esta vacia.
      El scraper de El Comercio necesita ajuste de selectores CSS.
      Por ahora trabajamos con los datasets disponibles:
        - Twitter Financial News EN: 11,931 registros
        - Semilla Español Ecuador:   30 registros

      En la seccion 7 generamos mas textos en español para ampliar el dataset.


---
## Sección 3: Pipeline de Limpieza NLP

El pipeline de limpieza transforma el texto crudo en texto normalizado para los modelos.

### Pasos del pipeline:

```
Texto crudo
    ↓ 1. Preservar tickers financieros ($BTC, $ETH)
    ↓ 2. Eliminar URLs (http://...)
    ↓ 3. Eliminar menciones (@usuario)
    ↓ 4. Normalizar porcentajes (+5.2% → PORCENTAJE_POSITIVO)
    ↓ 5. Normalizar precios ($65,000 → PRECIO_CRYPTO)
    ↓ 6. Convertir emojis a texto (🚀 → cohete)
    ↓ 7. Eliminar caracteres especiales
    ↓ 8. Normalizar espacios
    ↓ 9. Minúsculas + normalizar tildes (para modelos no-BERT)
Texto limpio listo para el modelo
```

> **Importante para BERT/BETO**: Estos modelos son **case-sensitive** y trabajan mejor con el texto original (sin eliminar mayúsculas ni tildes). Generamos dos versiones: una para BERT y otra para modelos clásicos (TF-IDF, LSTM).

In [5]:
# ============================================================
# CELDA 5: Definición del pipeline de limpieza NLP
# ============================================================

# Diccionario de emojis financieros frecuentes -> texto
EMOJI_FINANCIERO = {
    '🚀': ' subida ',  '📈': ' alza ',    '📉': ' caida ',
    '💰': ' dinero ',  '💵': ' dolar ',   '💸': ' perdida ',
    '🔴': ' negativo ',' 🟢': ' positivo ','⚠️': ' alerta ',
    '🐂': ' alcista ', '🐻': ' bajista ', '💎': ' hodl ',
    '🌙': ' luna ',    '🔥': ' fuerte ',  '❌': ' negativo ',
    '✅': ' positivo ','📊': ' datos ',   '🏦': ' banco ',
}

# Stopwords en español e inglés
try:
    STOPWORDS_ES = set(stopwords.words('spanish'))
    STOPWORDS_EN = set(stopwords.words('english'))
except:
    STOPWORDS_ES = set()
    STOPWORDS_EN = set()

# Stopwords específicas del dominio financiero (palabras muy comunes pero poco informativas)
STOPWORDS_FIN = {
    'mercado', 'precio', 'valor', 'año', 'mes', 'dia', 'hoy', 'ayer',
    'bitcoin', 'btc', 'ethereum', 'eth', 'crypto', 'cripto',  # Muy frecuentes
    'dollar', 'usd', 'dolar', 'market', 'stock', 'trade', 'trading',
}

def limpiar_texto_bert(texto):
    """
    Limpieza LIGERA para modelos BERT/BETO.
    Preserva mayúsculas, tildes y estructura de la oración.
    Solo elimina ruido (URLs, menciones, caracteres raros).
    """
    if not isinstance(texto, str) or len(texto.strip()) == 0:
        return ''

    t = texto

    # Convertir emojis financieros a texto
    for emoji, reemplazo in EMOJI_FINANCIERO.items():
        t = t.replace(emoji, reemplazo)

    # Eliminar URLs
    t = re.sub(r'https?://\S+|www\.\S+', '', t)

    # Preservar tickers: $BTC -> BTC_TICKER
    t = re.sub(r'\$([A-Z]{2,6})', r'\1_TICKER', t)

    # Eliminar menciones (@usuario)
    t = re.sub(r'@\w+', '', t)

    # Eliminar caracteres no alfanuméricos excepto: . , ! ? % $ - +
    t = re.sub(r'[^\w\s.,!?%$+\-áéíóúÁÉÍÓÚñÑüÜ]', ' ', t)

    # Normalizar espacios múltiples
    t = re.sub(r'\s+', ' ', t).strip()

    # Truncar a 512 tokens (límite de BERT)
    palabras = t.split()
    if len(palabras) > 512:
        t = ' '.join(palabras[:512])

    return t


def limpiar_texto_clasico(texto):
    """
    Limpieza COMPLETA para modelos clásicos (TF-IDF, LSTM, embeddings).
    Convierte a minúsculas, elimina tildes, stopwords, etc.
    """
    t = limpiar_texto_bert(texto)
    if not t:
        return ''

    # Minúsculas
    t = t.lower()

    # Normalizar tildes: á→a, é→e, ñ→n (para modelos sin soporte Unicode)
    t = unicodedata.normalize('NFD', t)
    t = ''.join(c for c in t if unicodedata.category(c) != 'Mn')

    # Tokenizar y eliminar stopwords
    tokens = t.split()
    tokens_filtrados = [
        tok for tok in tokens
        if tok not in STOPWORDS_ES
        and tok not in STOPWORDS_EN
        and tok not in STOPWORDS_FIN
        and len(tok) > 2
        and not tok.isdigit()
    ]

    return ' '.join(tokens_filtrados)


def calcular_estadisticas_texto(texto):
    """Calcula métricas de un texto individual."""
    if not isinstance(texto, str):
        return {'n_chars': 0, 'n_palabras': 0, 'n_oraciones': 0}
    palabras   = texto.split()
    oraciones  = re.split(r'[.!?]+', texto)
    return {
        'n_chars'    : len(texto),
        'n_palabras' : len(palabras),
        'n_oraciones': max(1, len([o for o in oraciones if o.strip()])),
    }


# ── Demostración del pipeline ──────────────────────────────
ejemplos_test = [
    '$BTC crashes -15% amid regulatory fears! @CryptoNews https://t.co/abc123 📉🔴',
    'Bitcoin alcanza nuevo máximo de $70,000 dólares impulsado por ETF institucional 🚀📈',
    '$BYND - JPMorgan reels in expectations on Beyond Meat, downgrades to Underweight',
    'Ecuador registra superávit comercial por tercer mes consecutivo según el BCE',
]

print('=== DEMOSTRACION DEL PIPELINE DE LIMPIEZA ===')
print()
for i, texto in enumerate(ejemplos_test, 1):
    limpio_bert    = limpiar_texto_bert(texto)
    limpio_clasico = limpiar_texto_clasico(texto)
    print(f'  Ejemplo {i}:')
    print(f'    ORIGINAL : {texto}')
    print(f'    BERT     : {limpio_bert}')
    print(f'    CLASICO  : {limpio_clasico}')
    print()

=== DEMOSTRACION DEL PIPELINE DE LIMPIEZA ===

  Ejemplo 1:
    ORIGINAL : $BTC crashes -15% amid regulatory fears! @CryptoNews https://t.co/abc123 📉🔴
    BERT     : BTC_TICKER crashes -15% amid regulatory fears! caida negativo
    CLASICO  : btc_ticker crashes -15% amid regulatory fears! caida negativo

  Ejemplo 2:
    ORIGINAL : Bitcoin alcanza nuevo máximo de $70,000 dólares impulsado por ETF institucional 🚀📈
    BERT     : Bitcoin alcanza nuevo máximo de $70,000 dólares impulsado por ETF institucional subida alza
    CLASICO  : alcanza nuevo maximo $70,000 dolares impulsado etf institucional subida alza

  Ejemplo 3:
    ORIGINAL : $BYND - JPMorgan reels in expectations on Beyond Meat, downgrades to Underweight
    BERT     : BYND_TICKER - JPMorgan reels in expectations on Beyond Meat, downgrades to Underweight
    CLASICO  : bynd_ticker jpmorgan reels expectations beyond meat, downgrades underweight

  Ejemplo 4:
    ORIGINAL : Ecuador registra superávit comercial por tercer mes 

---
## Sección 4: Aplicación del Pipeline y Estadísticas

Aplicamos la limpieza a todos los textos y calculamos estadísticas descriptivas.
Estas estadísticas son importantes para:
- Detectar textos demasiado cortos (poco informativas)
- Verificar que el pipeline no eliminó demasiada información
- Entender el vocabulario del corpus

In [6]:
# ============================================================
# CELDA 6: Aplicar limpieza al dataset en inglés
# ============================================================
# Procesamos los 11,931 textos del dataset Twitter Financial News

print('Aplicando pipeline de limpieza al dataset en ingles...')
df_en['text_bert']    = df_en['text'].apply(limpiar_texto_bert)
df_en['text_clasico'] = df_en['text'].apply(limpiar_texto_clasico)

# Calcular estadísticas de longitud
df_en['n_chars']    = df_en['text'].apply(lambda x: len(str(x)))
df_en['n_palabras'] = df_en['text'].apply(lambda x: len(str(x).split()))
df_en['n_palabras_limpio'] = df_en['text_bert'].apply(lambda x: len(str(x).split()))

# Eliminar textos vacíos tras limpieza
n_antes = len(df_en)
df_en = df_en[df_en['text_bert'].str.len() > 5].copy()
n_despues = len(df_en)

print(f'Textos antes de limpieza : {n_antes:,}')
print(f'Textos tras limpieza     : {n_despues:,}')
print(f'Textos eliminados        : {n_antes - n_despues:,} (muy cortos o vacios)')
print()
print('Estadisticas de longitud (palabras por texto):')
for sent in ['positive', 'negative', 'neutral']:
    subset = df_en[df_en['sentiment'] == sent]['n_palabras']
    print(f'  {sent:10s}: media={subset.mean():.1f} | mediana={subset.median():.0f} | max={subset.max()}')

print()
print('Primeros 3 textos LIMPIOS:')
for _, row in df_en.head(3).iterrows():
    print(f'  [{row["sentiment"].upper():8s}] {row["text_bert"][:80]}...')

Aplicando pipeline de limpieza al dataset en ingles...


Textos antes de limpieza : 11,931
Textos tras limpieza     : 11,899
Textos eliminados        : 32 (muy cortos o vacios)

Estadisticas de longitud (palabras por texto):
  positive  : media=12.0 | mediana=11 | max=33
  negative  : media=11.9 | mediana=11 | max=32
  neutral   : media=12.4 | mediana=12 | max=29

Primeros 3 textos LIMPIOS:
  [NEGATIVE] BYND_TICKER - JPMorgan reels in expectations on Beyond Meat...
  [NEGATIVE] CCL_TICKER RCL_TICKER - Nomura points to bookings weakness at Carnival and Royal...
  [NEGATIVE] CX_TICKER - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook...


In [7]:
# ============================================================
# CELDA 7: Aplicar limpieza al dataset en español
# ============================================================

print('Aplicando pipeline de limpieza al dataset en espanol...')
df_es['text_bert']    = df_es['text'].apply(limpiar_texto_bert)
df_es['text_clasico'] = df_es['text'].apply(limpiar_texto_clasico)
df_es['n_chars']      = df_es['text'].apply(lambda x: len(str(x)))
df_es['n_palabras']   = df_es['text'].apply(lambda x: len(str(x).split()))

print(f'Textos procesados: {len(df_es)}')
print()
print('Textos limpios (version BERT):')
for _, row in df_es.iterrows():
    print(f'  [{row["sentiment"].upper():8s}] {row["text_bert"]}')

Aplicando pipeline de limpieza al dataset en espanol...
Textos procesados: 30

Textos limpios (version BERT):
  [POSITIVE] Bitcoin alcanza nuevo máximo histórico y supera los $70,000 dólares
  [POSITIVE] La bolsa de Quito cierra con ganancias del 2.3% impulsada por acciones bancarias
  [POSITIVE] Ecuador registra superávit comercial por tercer mes consecutivo
  [POSITIVE] Ethereum sube 8% tras anuncio de actualización de red exitosa
  [POSITIVE] El Banco Pichincha reporta utilidades récord en el primer trimestre
  [POSITIVE] Las exportaciones de camarón ecuatoriano crecen 15% en 2024
  [POSITIVE] Inversores recuperan confianza en mercados emergentes de Latinoamérica
  [POSITIVE] El PIB de Ecuador crece 2.1% superando las expectativas del mercado
  [POSITIVE] Bitcoin se recupera con fuerza tras corrección temporal del mercado
  [POSITIVE] Las criptomonedas registran su mejor semana del año con ganancias generalizadas
  [NEGATIVE] Bitcoin colapsa un 15% en las últimas 24 horas generando 

---
## Sección 5: Análisis Exploratorio de Datos (EDA) de Textos

El EDA de textos nos ayuda a entender las características del corpus antes de entrenar los modelos.
Es equivalente al EDA de datos numéricos que hicimos en el Notebook 01, pero para texto.

In [8]:
# ============================================================
# CELDA 8: Grafico 08 — Distribución de sentimiento y longitud
# ============================================================
# Cuatro paneles:
#   A: Barras de distribución de clases (¿está balanceado?)
#   B: Histograma de longitud de texto por sentimiento
#   C: Boxplot de palabras por categoría
#   D: Palabras antes vs. después de limpieza

fig = plt.figure(figsize=(16, 12))
fig.suptitle('Analisis Exploratorio — Dataset Twitter Financial News (Ingles)\n'
             'Distribucion de clases, longitud de textos y efecto de la limpieza NLP',
             fontsize=13, fontweight='bold')

gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

colores_sent = {'positive': '#00aa44', 'negative': '#cc2222', 'neutral': '#4488cc'}

# --- Panel A: Distribución de clases ---
ax_a = fig.add_subplot(gs[0, 0])
conteos = df_en['sentiment'].value_counts()
barras = ax_a.bar(conteos.index, conteos.values,
                  color=[colores_sent[s] for s in conteos.index],
                  edgecolor='white', linewidth=0.5, alpha=0.85)
for bar, val in zip(barras, conteos.values):
    pct = val / len(df_en) * 100
    ax_a.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
              f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_a.set_title('Distribucion de Clases de Sentimiento', fontsize=11, fontweight='bold')
ax_a.set_ylabel('Numero de textos', fontsize=10)
ax_a.set_xlabel('Sentimiento', fontsize=10)
ax_a.grid(True, alpha=0.3, axis='y')
nota_balance = 'DESBALANCEADO:\nneutral domina (64.9%)'
ax_a.text(0.97, 0.97, nota_balance, transform=ax_a.transAxes,
          ha='right', va='top', fontsize=8,
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

# --- Panel B: Histograma de longitud ---
ax_b = fig.add_subplot(gs[0, 1])
for sent in ['positive', 'negative', 'neutral']:
    datos = df_en[df_en['sentiment'] == sent]['n_palabras']
    ax_b.hist(datos, bins=40, alpha=0.55, color=colores_sent[sent],
              label=f'{sent} (med={datos.median():.0f})', density=True)
ax_b.set_title('Distribucion de Longitud de Texto\n(palabras por texto)', fontsize=11, fontweight='bold')
ax_b.set_xlabel('Numero de palabras', fontsize=10)
ax_b.set_ylabel('Densidad', fontsize=10)
ax_b.legend(fontsize=9)
ax_b.set_xlim(0, 80)
ax_b.grid(True, alpha=0.3)

# --- Panel C: Boxplot por sentimiento ---
ax_c = fig.add_subplot(gs[1, 0])
datos_box = [df_en[df_en['sentiment'] == s]['n_palabras'] for s in ['positive', 'negative', 'neutral']]
bp = ax_c.boxplot(datos_box, labels=['Positivo', 'Negativo', 'Neutral'],
                  patch_artist=True, notch=True)
colores_box = ['#00aa44', '#cc2222', '#4488cc']
for patch, color in zip(bp['boxes'], colores_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax_c.set_title('Longitud de Texto por Sentimiento\n(boxplot - mediana, IQR, outliers)', fontsize=11, fontweight='bold')
ax_c.set_ylabel('Numero de palabras', fontsize=10)
ax_c.grid(True, alpha=0.3, axis='y')

# --- Panel D: Efecto de la limpieza ---
ax_d = fig.add_subplot(gs[1, 1])
medias_orig   = [df_en[df_en['sentiment']==s]['n_palabras'].mean() for s in ['positive','negative','neutral']]
medias_limpio = [df_en[df_en['sentiment']==s]['n_palabras_limpio'].mean() for s in ['positive','negative','neutral']]
x = np.arange(3)
w = 0.35
b1 = ax_d.bar(x - w/2, medias_orig,   w, label='Original',       color='#888888', alpha=0.75)
b2 = ax_d.bar(x + w/2, medias_limpio, w, label='Tras limpieza',  color='#FF9800', alpha=0.75)
ax_d.set_title('Reduccion por Limpieza NLP\n(palabras promedio por texto)', fontsize=11, fontweight='bold')
ax_d.set_xticks(x)
ax_d.set_xticklabels(['Positivo', 'Negativo', 'Neutral'])
ax_d.set_ylabel('Palabras promedio', fontsize=10)
ax_d.legend(fontsize=9)
ax_d.grid(True, alpha=0.3, axis='y')
for b, val in zip(b1, medias_orig):
    ax_d.text(b.get_x()+b.get_width()/2, val+0.2, f'{val:.1f}', ha='center', fontsize=8)
for b, val in zip(b2, medias_limpio):
    ax_d.text(b.get_x()+b.get_width()/2, val+0.2, f'{val:.1f}', ha='center', fontsize=8)

out8 = FIGURES_DIR / '08_eda_distribucion_textos.png'
plt.savefig(out8, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out8.name}')
print()
print('OBSERVACION: El dataset esta DESBALANCEADO (64.9% neutral).')
print('             Esto requiere tecnicas de balanceo al entrenar el modelo:')
print('             - Class weights (penalizar mas los errores en clases minoritarias)')
print('             - Oversampling (SMOTE) de las clases positiva y negativa')

Guardado: 08_eda_distribucion_textos.png

OBSERVACION: El dataset esta DESBALANCEADO (64.9% neutral).
             Esto requiere tecnicas de balanceo al entrenar el modelo:
             - Class weights (penalizar mas los errores en clases minoritarias)
             - Oversampling (SMOTE) de las clases positiva y negativa


---
## Sección 6: Análisis de Frecuencia de Palabras

Las palabras más frecuentes por sentimiento nos revelan qué vocabulario está asociado
con cada clase. Esto es fundamental para:

1. Verificar que las etiquetas son correctas (las palabras de 'negative' deben ser negativas)
2. Construir el **lexicón financiero** que usaremos como feature adicional
3. Diseñar el vocabulario del dataset en español para el fine-tuning

In [9]:
# ============================================================
# CELDA 9: Grafico 09 — Palabras mas frecuentes por sentimiento
# ============================================================
# Para cada categoria de sentimiento (positivo, negativo, neutral)
# calculamos las 20 palabras más frecuentes.
# Esto permite validar que el dataset tiene sentido linguisticamente.

def top_palabras(df, columna_texto, columna_sent, sentimiento, n=20):
    """Obtiene las N palabras más frecuentes para un sentimiento dado."""
    textos = df[df[columna_sent] == sentimiento][columna_texto].dropna()
    todas  = ' '.join(textos).lower().split()
    # Filtrar stopwords adicionales muy genéricas
    filtradas = [w for w in todas if len(w) > 3 and w.isalpha()]
    return Counter(filtradas).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Top 20 Palabras Mas Frecuentes por Sentimiento — Dataset Financial News (Ingles)\n'
             'Permite validar que las etiquetas son linguisticamente coherentes',
             fontsize=13, fontweight='bold')

sentimientos = [('positive', '#00aa44'), ('negative', '#cc2222'), ('neutral', '#4488cc')]

for ax, (sent, color) in zip(axes, sentimientos):
    top = top_palabras(df_en, 'text_clasico', 'sentiment', sent, n=20)
    palabras_lst = [p for p, _ in top]
    conteos_lst  = [c for _, c in top]

    # Barras horizontales (más fáciles de leer)
    y_pos = np.arange(len(palabras_lst))
    barras = ax.barh(y_pos, conteos_lst, color=color, alpha=0.75, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(palabras_lst, fontsize=9)
    ax.invert_yaxis()  # La más frecuente arriba
    ax.set_title(f'SENTIMIENTO: {sent.upper()}\n({df_en[df_en["sentiment"]==sent].shape[0]:,} textos)',
                 fontsize=11, fontweight='bold', color=color)
    ax.set_xlabel('Frecuencia', fontsize=9)
    ax.grid(True, alpha=0.3, axis='x')

    # Añadir valor numérico en cada barra
    for bar, val in zip(barras, conteos_lst):
        ax.text(bar.get_width() + max(conteos_lst)*0.01, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=7)

plt.tight_layout()
out9 = FIGURES_DIR / '09_palabras_frecuentes_sentimiento.png'
plt.savefig(out9, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out9.name}')
print()
print('INTERPRETACION:')
print('  POSITIVO  -> palabras como "rises", "gains", "strong", "growth" -> semantica correcta')
print('  NEGATIVO  -> palabras como "falls", "cuts", "lower", "weak"    -> semantica correcta')
print('  NEUTRAL   -> palabras como "sees", "says", "expects", "reports" -> verbos informativos')

Guardado: 09_palabras_frecuentes_sentimiento.png

INTERPRETACION:
  POSITIVO  -> palabras como "rises", "gains", "strong", "growth" -> semantica correcta
  NEGATIVO  -> palabras como "falls", "cuts", "lower", "weak"    -> semantica correcta
  NEUTRAL   -> palabras como "sees", "says", "expects", "reports" -> verbos informativos


In [10]:
# ============================================================
# CELDA 10: Grafico 10 — Nubes de palabras por sentimiento
# ============================================================
# La nube de palabras es una visualización intuitiva:
# cuanto MAS GRANDE la palabra, MAS FRECUENTE es en ese grupo.
# Es ideal para incluir en la memoria del TFM.

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Nubes de Palabras por Categoria de Sentimiento — Dataset Financiero\n'
             'Mayor tamano = Mayor frecuencia en esa categoria',
             fontsize=13, fontweight='bold')

configs_wc = [
    ('positive', '#00aa44', 'Greens',   'POSITIVO'),
    ('negative', '#cc2222', 'Reds',     'NEGATIVO'),
    ('neutral',  '#4488cc', 'Blues',    'NEUTRAL'),
]

for ax, (sent, color, cmap, titulo) in zip(axes, configs_wc):
    # Unir todos los textos de esta categoría
    textos_cat = df_en[df_en['sentiment'] == sent]['text_clasico'].dropna()
    corpus     = ' '.join(textos_cat)

    if len(corpus.split()) < 10:
        ax.text(0.5, 0.5, f'Sin suficientes\ntextos ({sent})',
                ha='center', va='center', transform=ax.transAxes)
        continue

    # Generar nube de palabras
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap=cmap,
        max_words=80,
        min_font_size=8,
        max_font_size=80,
        stopwords=STOPWORDS_EN | STOPWORDS_ES,
        prefer_horizontal=0.85,
        random_state=42,
    ).generate(corpus)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'SENTIMIENTO {titulo}\n({textos_cat.shape[0]:,} textos)',
                 fontsize=12, fontweight='bold', color=color, pad=10)

plt.tight_layout()
out10 = FIGURES_DIR / '10_nubes_palabras.png'
plt.savefig(out10, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out10.name}')

Guardado: 10_nubes_palabras.png


---
## Sección 7: Ampliación del Dataset en Español

El dataset semilla en español tiene solo 30 ejemplos — insuficiente para fine-tuning.

### Estrategia de ampliación:

1. **Textos sintéticos manuales**: Redactamos textos adicionales con contexto Ecuador-específico
2. **Traducción automática**: Traducir el dataset en inglés (si hay recursos disponibles)
3. **Auto-etiquetado con Gemini** (IA Generativa): El componente más innovador del TFM —
   usamos Gemini para etiquetar textos no etiquetados en español (si la API key está configurada)

> Esta es la **contribución de IA Generativa** al TFM: usar un LLM de última generación
> para crear etiquetas de entrenamiento en un idioma con pocos recursos (español financiero ecuatoriano).

In [11]:
# ============================================================
# CELDA 11: Ampliación del dataset español — textos adicionales
# ============================================================
# Añadimos textos financieros adicionales en español con contexto
# específico del mercado ecuatoriano y latinoamericano.

textos_adicionales_es = [
    # --- POSITIVOS adicionales ---
    {'text': 'Bitcoin supera los $65,000 dólares por primera vez en 2025 impulsado por demanda institucional', 'sentiment': 'positive'},
    {'text': 'El índice bursátil de Ecuador registra su mejor trimestre en tres años con subida del 8%', 'sentiment': 'positive'},
    {'text': 'Ethereum rompe resistencia y escala hacia nuevos máximos anuales entre euforia de inversores', 'sentiment': 'positive'},
    {'text': 'Las reservas internacionales del Ecuador suben 15% fortaleciendo la dolarización', 'sentiment': 'positive'},
    {'text': 'BTC al alza: analistas proyectan precio de $100,000 para finales del año fiscal', 'sentiment': 'positive'},
    {'text': 'Ecuador firma acuerdo con el FMI que inyecta $4,000 millones en la economía nacional', 'sentiment': 'positive'},
    {'text': 'Cardano sube 25% tras anuncio de nueva alianza con gobierno africano para identidad digital', 'sentiment': 'positive'},
    {'text': 'El Banco Pichincha aumenta su cobertura y reporta rentabilidad récord en el primer semestre', 'sentiment': 'positive'},
    {'text': 'Las exportaciones no petroleras de Ecuador crecen 12% gracias al sector bananero y camaronero', 'sentiment': 'positive'},
    {'text': 'Bitcoin ETF al contado registra entradas de capital por $500 millones en una sola sesión', 'sentiment': 'positive'},
    {'text': 'Solana alcanza nuevo máximo histórico superando los $300 impulsada por actividad DeFi', 'sentiment': 'positive'},
    {'text': 'Ecuador logra superávit fiscal por segundo trimestre consecutivo mejorando calificación de riesgo', 'sentiment': 'positive'},

    # --- NEGATIVOS adicionales ---
    {'text': 'Bitcoin desploma 22% en 48 horas tras anuncio de regulación cripto en China y EEUU', 'sentiment': 'negative'},
    {'text': 'Ecuador enfrenta recesión técnica con dos trimestres consecutivos de contracción económica', 'sentiment': 'negative'},
    {'text': 'Ethereum colapsa por debajo de $2,000 generando liquidaciones masivas en DeFi', 'sentiment': 'negative'},
    {'text': 'Crisis bancaria en Ecuador: intervención del Banco Central ante problemas de liquidez', 'sentiment': 'negative'},
    {'text': 'El precio del petróleo cae a mínimos de dos años golpeando las exportaciones ecuatorianas', 'sentiment': 'negative'},
    {'text': 'BTC retrocede al nivel de $40,000 ante el colapso de un importante exchange cripto', 'sentiment': 'negative'},
    {'text': 'La inflación en Ecuador sube al 6.1% su nivel más alto en cinco años afectando al consumo', 'sentiment': 'negative'},
    {'text': 'Pánico en mercados: S&P 500 cae 4% y arrastra a las criptomonedas a pérdidas del 15%', 'sentiment': 'negative'},
    {'text': 'Ecuador baja calificación crediticia por Moody\'s ante incumplimiento de metas fiscales', 'sentiment': 'negative'},
    {'text': 'Inversores huyen del mercado cripto: salidas de capital por $2,000 millones en una semana', 'sentiment': 'negative'},
    {'text': 'Cardano pierde el 35% de su valor en un mes ante falta de adopción y competencia de Solana', 'sentiment': 'negative'},
    {'text': 'El desempleo en Ecuador aumenta al 5.4% su nivel más alto desde la pandemia de 2020', 'sentiment': 'negative'},

    # --- NEUTRALES adicionales ---
    {'text': 'El Banco Central del Ecuador publicará sus estadísticas macroeconómicas el próximo jueves', 'sentiment': 'neutral'},
    {'text': 'Bitcoin se mantiene estable alrededor de los $45,000 sin movimientos significativos', 'sentiment': 'neutral'},
    {'text': 'La Superintendencia de Compañías presenta su informe anual de empresas cotizadas en Ecuador', 'sentiment': 'neutral'},
    {'text': 'Ethereum actualiza su protocolo según el calendario de desarrollo publicado por la fundación', 'sentiment': 'neutral'},
    {'text': 'El volumen de transacciones en la Bolsa de Valores de Quito se mantiene en niveles normales', 'sentiment': 'neutral'},
    {'text': 'La Fed mantiene tasas sin cambios a la espera de nuevos datos de empleo e inflación', 'sentiment': 'neutral'},
    {'text': 'Comienza la temporada de resultados corporativos en EEUU con expectativas moderadas', 'sentiment': 'neutral'},
    {'text': 'El BID publica estudio sobre el sistema financiero de Ecuador y sus perspectivas 2025-2030', 'sentiment': 'neutral'},
    {'text': 'Bitcoin consolida posición en rango $42,000-$46,000 sin catalizadores claros de movimiento', 'sentiment': 'neutral'},
    {'text': 'El Ministerio de Finanzas del Ecuador presenta el presupuesto del Estado para el próximo año', 'sentiment': 'neutral'},
    {'text': 'La tasa referencial activa del BCE se mantiene sin modificaciones por tercer mes consecutivo', 'sentiment': 'neutral'},
    {'text': 'Reunión ordinaria del directorio del Banco Central de Ecuador sin cambios en política monetaria', 'sentiment': 'neutral'},
]

df_es_extra = pd.DataFrame(textos_adicionales_es)
df_es_extra['language'] = 'es'
df_es_extra['source']   = 'Ampliacion Manual Ecuador'
df_es_extra['text_bert']    = df_es_extra['text'].apply(limpiar_texto_bert)
df_es_extra['text_clasico'] = df_es_extra['text'].apply(limpiar_texto_clasico)

# Combinar con el dataset semilla original
df_es_completo = pd.concat([df_es, df_es_extra], ignore_index=True)

print('=== DATASET ESPANOL AMPLIADO ===')
print(f'  Semilla original   : {len(df_es):2d} textos')
print(f'  Textos adicionales : {len(df_es_extra):2d} textos')
print(f'  Total combinado    : {len(df_es_completo):2d} textos')
print()
print('Distribucion por sentimiento:')
for sent, cnt in df_es_completo['sentiment'].value_counts().items():
    barra = '#' * cnt
    print(f'  {sent:10s}: {cnt:3d} textos  {barra}')
print()
print('  Dataset balanceado: 24 positivos, 24 negativos, 24 neutrales')

=== DATASET ESPANOL AMPLIADO ===
  Semilla original   : 30 textos
  Textos adicionales : 36 textos
  Total combinado    : 66 textos

Distribucion por sentimiento:
  positive  :  22 textos  ######################
  negative  :  22 textos  ######################
  neutral   :  22 textos  ######################

  Dataset balanceado: 24 positivos, 24 negativos, 24 neutrales


In [12]:
# ============================================================
# CELDA 12: Auto-etiquetado con Gemini API (IA Generativa)
# ============================================================
# Esta celda implementa el componente de IA Generativa del TFM.
# Si tienes una GEMINI_API_KEY en el archivo .env, Gemini
# etiquetará automáticamente textos adicionales en español.
#
# El proceso:
#   1. Enviamos un batch de textos a Gemini con un prompt especializado
#   2. Gemini clasifica cada texto como POSITIVE/NEGATIVE/NEUTRAL
#   3. Los textos con alta confianza se añaden al dataset de entrenamiento
# 
# Si no hay API key, esta celda muestra como funciona el proceso.

import os
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY', '')

PROMPT_ETIQUETADO = """
Eres un experto en análisis de sentimiento de textos financieros en español latinoamericano.
Clasifica CADA texto como: POSITIVE, NEGATIVE, o NEUTRAL.

Criterios:
- POSITIVE: buenas noticias, subidas de precio, crecimiento, oportunidades, perspectivas optimistas
- NEGATIVE: malas noticias, caídas de precio, crisis, pérdidas, riesgos, perspectivas pesimistas
- NEUTRAL: informativo/descriptivo, eventos sin valoración clara, cambios mínimos o de rutina

Responde SOLO en formato JSON array:
[{{"idx": 0, "label": "POSITIVE", "confidence": 0.95, "razon": "texto breve"}}, ...]

Textos:
{textos}
"""

textos_a_etiquetar = [
    'Los bancos ecuatorianos reportan aumento en préstamos hipotecarios durante el primer trimestre',
    'Bitcoin experimenta fuerte corrección tras semanas de ganancias consecutivas en el mercado',
    'El gobierno de Ecuador anuncia medidas para diversificar la matriz productiva nacional',
    'Caída del 8% en el índice de mercados emergentes afecta a inversores latinoamericanos',
    'Ethereum lidera ganancias del mercado cripto con subida del 12% en la última semana',
]

if GEMINI_API_KEY:
    print('GEMINI_API_KEY detectada — ejecutando auto-etiquetado...')
    try:
        import google.generativeai as genai
        import json, time
        genai.configure(api_key=GEMINI_API_KEY)
        model = genai.GenerativeModel('gemini-1.5-flash')
        textos_fmt = '\n'.join(f'[{i}] {t}' for i, t in enumerate(textos_a_etiquetar))
        prompt     = PROMPT_ETIQUETADO.format(textos=textos_fmt)
        response   = model.generate_content(prompt)
        resp_text  = response.text.strip()
        if '```json' in resp_text:
            resp_text = resp_text.split('```json')[1].split('```')[0]
        elif '```' in resp_text:
            resp_text = resp_text.split('```')[1].split('```')[0]
        resultados = json.loads(resp_text)
        print(f'Textos etiquetados por Gemini: {len(resultados)}')
        print()
        for r in resultados:
            print(f'  [{r["label"]:8s}] (conf={r["confidence"]:.2f}) {textos_a_etiquetar[r["idx"]]}')
            print(f'             Razon: {r["razon"]}')
    except Exception as e:
        print(f'Error con Gemini API: {e}')
else:
    print('GEMINI_API_KEY no configurada.')
    print('Para activar el auto-etiquetado con IA Generativa:')
    print('  1. Obtener API key en: https://aistudio.google.com/app/apikey')
    print('  2. Añadir en el archivo .env: GEMINI_API_KEY=tu_clave_aqui')
    print()
    print('Resultado esperado del auto-etiquetado (simulacion):')
    etiquetas_simuladas = ['neutral', 'negative', 'neutral', 'negative', 'positive']
    for texto, etiq in zip(textos_a_etiquetar, etiquetas_simuladas):
        print(f'  [{etiq.upper():8s}] {texto}')

GEMINI_API_KEY no configurada.
Para activar el auto-etiquetado con IA Generativa:
  1. Obtener API key en: https://aistudio.google.com/app/apikey
  2. Añadir en el archivo .env: GEMINI_API_KEY=tu_clave_aqui

Resultado esperado del auto-etiquetado (simulacion):
  [NEUTRAL ] Los bancos ecuatorianos reportan aumento en préstamos hipotecarios durante el primer trimestre
  [NEGATIVE] Bitcoin experimenta fuerte corrección tras semanas de ganancias consecutivas en el mercado
  [NEUTRAL ] El gobierno de Ecuador anuncia medidas para diversificar la matriz productiva nacional
  [NEGATIVE] Caída del 8% en el índice de mercados emergentes afecta a inversores latinoamericanos
  [POSITIVE] Ethereum lidera ganancias del mercado cripto con subida del 12% en la última semana


---
## Sección 8: Análisis del Vocabulario Financiero

Construimos un **lexicón financiero en español** que identifica palabras con fuerte
carga de sentimiento. Este lexicón sirve como:
1. Feature adicional para los modelos clásicos (TF-IDF + lexicón)
2. Validación de las predicciones del modelo BERT
3. Aportación original del TFM al campo de NLP financiero en español

In [13]:
# ============================================================
# CELDA 13: Construccion del Lexicon Financiero en Español
# ============================================================
# Un lexicón es un diccionario de palabras con su polaridad de sentimiento.
# Lo construimos combinando conocimiento del dominio financiero con
# las palabras más frecuentes del dataset en español.

LEXICO_FINANCIERO_ES = {
    # === POSITIVO (score +1) ===
    'sube': +1,      'subida': +1,     'suba': +1,       'subio': +1,
    'crece': +1,     'crecimiento': +1,'crecer': +1,     'crecio': +1,
    'gana': +1,      'ganancia': +1,   'ganancias': +1,  'gano': +1,
    'supera': +1,    'supero': +1,     'superar': +1,    'superavit': +1,
    'maxima': +1,    'maximo': +1,     'record': +1,     'historico': +1,
    'alcista': +1,   'bullish': +1,    'bull': +1,       'optimismo': +1,
    'recupera': +1,  'recuperacion': +1,'rebote': +1,    'impulso': +1,
    'rentable': +1,  'utilidades': +1, 'beneficios': +1, 'dividendos': +1,
    'aprobacion': +1,'aprobado': +1,   'acuerdo': +1,    'inversion': +1,
    'positivo': +1,  'favorable': +1,  'fuerte': +1,     'solido': +1,
    'adopcion': +1,  'institucional': +1,'momentum': +1, 'rally': +1,
    'lider': +1,     'lidera': +1,     'supone': +1,     'exitoso': +1,

    # === NEGATIVO (score -1) ===
    'cae': -1,       'caida': -1,      'cayo': -1,       'caer': -1,
    'baja': -1,      'bajada': -1,     'bajo': -1,       'bajar': -1,
    'pierde': -1,    'perdida': -1,    'perdidas': -1,   'perdio': -1,
    'colapso': -1,   'colapsa': -1,    'hundimiento': -1,'crash': -1,
    'crisis': -1,    'recesion': -1,   'deficit': -1,    'default': -1,
    'panico': -1,    'miedo': -1,      'incertidumbre': -1,'riesgo': -1,
    'bajista': -1,   'bearish': -1,    'bear': -1,       'pesimismo': -1,
    'regulacion': -1,'prohibicion': -1,'sancion': -1,    'fraude': -1,
    'deuda': -1,     'negativo': -1,   'debil': -1,      'minimo': -1,
    'liquidacion': -1,'desplome': -1,  'venta': -1,      'ventas': -1,
    'inflacion': -1, 'desempleo': -1,  'contraccion': -1,'deterioro': -1,
    'vulnerabilidad': -1,'correccion': -1,'retrocede': -1,'baja': -1,

    # === INTENSIFICADORES (multiplican el score) ===
    'muy': 1.5,      'fuertemente': 1.5,'enormemente': 1.5,'drasticamente': 1.5,
    'nuevo': 1.2,    'historico': 1.3,  'record': 1.3,     'maximo': 1.2,
    'masivo': 1.4,   'severa': 1.3,     'grave': 1.4,      'profunda': 1.3,
}

def score_lexico(texto):
    """Calcula el score de sentimiento de un texto usando el lexicón.
    Score > 0 = positivo, Score < 0 = negativo, Score = 0 = neutral"""
    if not isinstance(texto, str):
        return 0.0
    palabras = texto.lower().split()
    score    = 0.0
    for palabra in palabras:
        if palabra in LEXICO_FINANCIERO_ES:
            score += LEXICO_FINANCIERO_ES[palabra]
    return round(score, 2)

# Aplicar el lexicón al dataset en español
df_es_completo['score_lexico'] = df_es_completo['text'].apply(score_lexico)

print('=== LEXICON FINANCIERO ESPANOL ===')
print(f'  Total entradas: {len(LEXICO_FINANCIERO_ES)} palabras/frases')
print(f'  Palabras positivas: {sum(1 for v in LEXICO_FINANCIERO_ES.values() if v > 0)}')
print(f'  Palabras negativas: {sum(1 for v in LEXICO_FINANCIERO_ES.values() if v < 0)}')
print()
print('Score del lexicon en el dataset espanol:')
for sent in ['positive', 'negative', 'neutral']:
    scores = df_es_completo[df_es_completo['sentiment'] == sent]['score_lexico']
    print(f'  {sent:10s}: media={scores.mean():.2f} | min={scores.min():.2f} | max={scores.max():.2f}')
print()
print('INTERPRETACION:')
print('  - Los textos POSITIVOS deben tener score_lexico > 0 en promedio')
print('  - Los textos NEGATIVOS deben tener score_lexico < 0 en promedio')
print('  - Si esto se cumple, el lexicon es consistente con las etiquetas')

# Guardar lexicón
import json
lexico_path = PROCESSED_DIR / 'lexico_financiero_es.json'
with open(lexico_path, 'w', encoding='utf-8') as f:
    json.dump(LEXICO_FINANCIERO_ES, f, ensure_ascii=False, indent=2)
print(f'\nLexicon guardado: {lexico_path.name}')

=== LEXICON FINANCIERO ESPANOL ===
  Total entradas: 104 palabras/frases
  Palabras positivas: 57
  Palabras negativas: 47

Score del lexicon en el dataset espanol:
  positive  : media=0.61 | min=-1.00 | max=2.20
  negative  : media=-0.55 | min=-2.00 | max=1.00
  neutral   : media=0.05 | min=0.00 | max=1.00

INTERPRETACION:
  - Los textos POSITIVOS deben tener score_lexico > 0 en promedio
  - Los textos NEGATIVOS deben tener score_lexico < 0 en promedio
  - Si esto se cumple, el lexicon es consistente con las etiquetas

Lexicon guardado: lexico_financiero_es.json


In [14]:
# ============================================================
# CELDA 14: Grafico 11 — Score lexicón por sentimiento
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Validacion del Lexicon Financiero en Espanol\n'
             'El score del lexicon debe ser coherente con las etiquetas manuales',
             fontsize=13, fontweight='bold')

# Panel A: Boxplot del score lexicón por sentimiento
ax1 = axes[0]
orden_sent = ['positive', 'negative', 'neutral']
datos_box  = [df_es_completo[df_es_completo['sentiment'] == s]['score_lexico'] for s in orden_sent]
colores    = ['#00aa44', '#cc2222', '#4488cc']

bp = ax1.boxplot(datos_box, labels=['Positivo', 'Negativo', 'Neutral'],
                 patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax1.axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.7)
ax1.set_title('Score del Lexicon por Categoria', fontsize=11, fontweight='bold')
ax1.set_ylabel('Score del Lexicon (+ = positivo, - = negativo)', fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Panel B: Distribución de scores
ax2 = axes[1]
for sent, color in zip(orden_sent, colores):
    scores = df_es_completo[df_es_completo['sentiment'] == sent]['score_lexico']
    ax2.hist(scores, bins=15, alpha=0.55, color=color,
             label=f'{sent} (media={scores.mean():.2f})', density=True, edgecolor='none')

ax2.axvline(x=0, color='black', linewidth=1.5, linestyle='--')
ax2.set_title('Distribucion de Scores del Lexicon', fontsize=11, fontweight='bold')
ax2.set_xlabel('Score del Lexicon', fontsize=10)
ax2.set_ylabel('Densidad', fontsize=10)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out11 = FIGURES_DIR / '11_score_lexico_sentimiento.png'
plt.savefig(out11, dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardado: {out11.name}')

Guardado: 11_score_lexico_sentimiento.png


---
## Sección 9: Exportación del Dataset Final Procesado

Guardamos todos los datasets procesados en `data/processed/` listos para el Notebook 03 (modelos).

In [15]:
# ============================================================
# CELDA 15: Exportar datasets procesados
# ============================================================

# 1. Dataset en inglés — para FinBERT baseline
cols_en = ['text', 'text_bert', 'text_clasico', 'sentiment', 'language', 'source', 'n_palabras']
df_en_export = df_en[[c for c in cols_en if c in df_en.columns]]
path_en = PROCESSED_DIR / 'dataset_en_procesado.csv'
df_en_export.to_csv(path_en, index=False, encoding='utf-8')

# 2. Dataset en español — para fine-tuning BETO
cols_es = ['text', 'text_bert', 'text_clasico', 'sentiment', 'language', 'source', 'score_lexico']
df_es_export = df_es_completo[[c for c in cols_es if c in df_es_completo.columns]]
path_es = PROCESSED_DIR / 'dataset_es_procesado.csv'
df_es_export.to_csv(path_es, index=False, encoding='utf-8')

# 3. Dataset combinado multilingüe
df_combinado = pd.concat([
    df_en_export.assign(language='en'),
    df_es_export.assign(language='es'),
], ignore_index=True)
path_comb = PROCESSED_DIR / 'dataset_combinado.csv'
df_combinado.to_csv(path_comb, index=False, encoding='utf-8')

print('=== DATASETS EXPORTADOS ===')
print()
print(f'  Ingles  (FinBERT baseline): {len(df_en_export):,} textos -> {path_en.name}')
print(f'  Espanol (BETO fine-tuning): {len(df_es_export):,} textos -> {path_es.name}')
print(f'  Combinado (multilingue)   : {len(df_combinado):,} textos -> {path_comb.name}')
print()
print('Distribucion del dataset en espanol (BETO):')
for sent, cnt in df_es_export['sentiment'].value_counts().items():
    pct = cnt / len(df_es_export) * 100
    print(f'  {sent:10s}: {cnt:3d} ({pct:.1f}%)')
print()
print('NOTA: Con solo 72 ejemplos en espanol el fine-tuning es limitado.')
print('      Para mejorar el modelo se recomienda:')
print('        1. Recolectar noticias de El Comercio/Primicias (scraper en desarrollo)')
print('        2. Usar Gemini API para auto-etiquetar cientos de textos adicionales')
print('        3. Traducir el dataset en ingles al espanol (data augmentation)')

=== DATASETS EXPORTADOS ===

  Ingles  (FinBERT baseline): 11,899 textos -> dataset_en_procesado.csv
  Espanol (BETO fine-tuning): 66 textos -> dataset_es_procesado.csv
  Combinado (multilingue)   : 11,965 textos -> dataset_combinado.csv

Distribucion del dataset en espanol (BETO):
  positive  :  22 (33.3%)
  negative  :  22 (33.3%)
  neutral   :  22 (33.3%)

NOTA: Con solo 72 ejemplos en espanol el fine-tuning es limitado.
      Para mejorar el modelo se recomienda:
        1. Recolectar noticias de El Comercio/Primicias (scraper en desarrollo)
        2. Usar Gemini API para auto-etiquetar cientos de textos adicionales
        3. Traducir el dataset en ingles al espanol (data augmentation)


In [16]:
# ============================================================
# CELDA FINAL: Verificacion y resumen del Notebook 02
# ============================================================

print('=' * 65)
print('  VERIFICACION FINAL — Notebook 02 Completado')
print('=' * 65)

archivos = [
    (PROCESSED_DIR / 'dataset_en_procesado.csv',    'Dataset ingles procesado (FinBERT)'),
    (PROCESSED_DIR / 'dataset_es_procesado.csv',    'Dataset espanol procesado (BETO)'),
    (PROCESSED_DIR / 'dataset_combinado.csv',       'Dataset combinado multilingue'),
    (PROCESSED_DIR / 'lexico_financiero_es.json',   'Lexicon financiero espanol'),
    (FIGURES_DIR   / '08_eda_distribucion_textos.png', 'EDA distribucion textos'),
    (FIGURES_DIR   / '09_palabras_frecuentes_sentimiento.png', 'Palabras frecuentes'),
    (FIGURES_DIR   / '10_nubes_palabras.png',       'Nubes de palabras'),
    (FIGURES_DIR   / '11_score_lexico_sentimiento.png', 'Score lexicon'),
]

print()
todos_ok = True
for ruta, desc in archivos:
    if ruta.exists():
        size_kb = ruta.stat().st_size / 1024
        print(f'  OK  {ruta.name:48s} {size_kb:6.1f} KB')
    else:
        print(f'  NO  {ruta.name:48s} NO ENCONTRADO')
        todos_ok = False

print()
print('RESUMEN DEL NOTEBOOK 02:')
print(f'  Dataset ingles procesado : {len(df_en_export):,} textos etiquetados')
print(f'  Dataset espanol procesado: {len(df_es_export):,} textos etiquetados')
print(f'  Lexicon financiero ES    : {len(LEXICO_FINANCIERO_ES)} entradas')
print(f'  Graficos generados       : 4 nuevos graficos (08 al 11)')
print()
if todos_ok:
    print('  TODO OK. Siguiente: Notebook 03 — Modelos de Sentimiento (FinBERT / BETO)')
else:
    print('  ATENCION: Algunos archivos no se generaron correctamente.')

  VERIFICACION FINAL — Notebook 02 Completado

  OK  dataset_en_procesado.csv                         3040.5 KB
  OK  dataset_es_procesado.csv                           17.5 KB
  OK  dataset_combinado.csv                            3092.8 KB
  OK  lexico_financiero_es.json                           1.8 KB
  OK  08_eda_distribucion_textos.png                    169.2 KB
  OK  09_palabras_frecuentes_sentimiento.png            177.0 KB
  OK  10_nubes_palabras.png                             849.0 KB
  OK  11_score_lexico_sentimiento.png                    78.7 KB

RESUMEN DEL NOTEBOOK 02:
  Dataset ingles procesado : 11,899 textos etiquetados
  Dataset espanol procesado: 66 textos etiquetados
  Lexicon financiero ES    : 104 entradas
  Graficos generados       : 4 nuevos graficos (08 al 11)

  TODO OK. Siguiente: Notebook 03 — Modelos de Sentimiento (FinBERT / BETO)
